# March Madness Predictor

Authors: Frank McLaughlin, Jonah Simonson  
UW Madison Sports Analytics Club March Madness Data Challenge 2026  

## Approach
- Combine three sources: KenPom/DEV efficiency stats, Kaggle March Mania game results, and Massey Ordinals rankings
- Build and clean a historical dataset covering 24 years of tournament data (2002–2025)
- Engineer 45 matchup-level features: efficiency differentials, shooting stats, assist/steal/block rates, win records, and KENPOM rankings
- Validate using leave-one-year-out cross-validation (LOO-CV)
- Tune a Random Forest with RandomizedSearchCV resulting in 0.1420 Brier score and 81.4% accuracy on past data
- Generate predictions for all 2278 possible 2026 tournament matchups


## 1. Setup & Data Loading

We use three Kaggle data sources:
- `jonathanpilafas/2024-march-madness-statistical-analysis` - KenPom/DEV efficiency stats
- `nishaanamin/march-madness-data` - submission template for 2026 matchups
- Kaggle March Mania 2026 - tournament game results, seeds, regular season results, and Massey Ordinals

In [3]:
#imports
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import brier_score_loss
import pickle

# Download datasets
path_dev = kagglehub.dataset_download("jonathanpilafas/2024-march-madness-statistical-analysis")
path_bart = kagglehub.dataset_download("nishaanamin/march-madness-data")
mania_path = os.path.expanduser('~/Parallels/march_madness/mania_data')


## 2. Load Data Sources

Datasets:
1. DEV dataset - KenPom/Barttorvik efficiency stats for every team, every year
2. Regular season results - to compute each team's win percentage heading into the tournament
3. Tournament results + seeds - the actual game outcomes we train on
4. Massey Ordinals - pre-tournament KENPOM rankings

In [5]:
#Load DEV dataset 
dev = pd.read_csv(os.path.join(path_dev, 'DEV _ March Madness.csv'))

#Load Mania data
mteams = pd.read_csv(os.path.join(mania_path, 'MTeams.csv'))
tourney_results = pd.read_csv(os.path.join(mania_path, 'MNCAATourneyCompactResults.csv'))
tourney_seeds = pd.read_csv(os.path.join(mania_path, 'MNCAATourneySeeds.csv'))
reg_season = pd.read_csv(os.path.join(mania_path, 'MRegularSeasonCompactResults.csv'))
massey = pd.read_csv(os.path.join(mania_path, 'MMasseyOrdinals.csv'))

#Load submission template
df_sub = pd.read_csv(os.path.expanduser('~/Parallels/march_madness/2026_Potential_Matchups.csv'))


# Build win percentage from regular season results
wins = reg_season.groupby(['Season','WTeamID']).size().reset_index(name='Wins')
losses = reg_season.groupby(['Season','LTeamID']).size().reset_index(name='Losses')
record = wins.merge(losses, left_on=['Season','WTeamID'], right_on=['Season','LTeamID'], how='outer')
record['TeamID'] = record['WTeamID'].fillna(record['LTeamID']).astype(int)
record['Wins'] = record['Wins'].fillna(0)
record['Losses'] = record['Losses'].fillna(0)
record['WinPct'] = record['Wins'] / (record['Wins'] + record['Losses'])
record = record[['Season','TeamID','WinPct']]


# Extract POM rankings
def get_latest_rankings(system):
    out = []
    for season in massey['Season'].unique():
        sub = massey[(massey['SystemName'] == system) & (massey['Season'] == season)]
        if len(sub) == 0: continue
        latest = sub[sub['RankingDayNum'] == sub['RankingDayNum'].max()][['Season','TeamID','OrdinalRank']].copy()
        latest.rename(columns={'OrdinalRank': f'{system}_Rank'}, inplace=True)
        out.append(latest)
    return pd.concat(out)

pom_ranks = get_latest_rankings('POM')


## 3. Build Game-Level Dataset

We construct one row per tournament game, framing it as higher seed vs lower seed. 

Target: Did the higher seed win? 1 = yes, 0 = no

In [7]:
# Clean seeds and build game level dataset using regex
tourney_seeds['SeedNum'] = tourney_seeds['Seed'].str.extract(r'(\d+)').astype(int) 
tourney_seeds_clean = tourney_seeds[['Season','TeamID','SeedNum']].copy()

games_mania = tourney_results[tourney_results['Season'] >= 2002].copy()

# Merge winner and loser seeds and team names
games_mania = games_mania.merge(
    tourney_seeds_clean.rename(columns={'TeamID':'WTeamID','SeedNum':'WSeed'}), on=['Season','WTeamID'], how='left')
games_mania = games_mania.merge(
    tourney_seeds_clean.rename(columns={'TeamID':'LTeamID','SeedNum':'LSeed'}), on=['Season','LTeamID'], how='left')
games_mania = games_mania.merge(
    mteams[['TeamID','TeamName']].rename(columns={'TeamID':'WTeamID','TeamName':'WTeamName'}), on='WTeamID', how='left')
games_mania = games_mania.merge(
    mteams[['TeamID','TeamName']].rename(columns={'TeamID':'LTeamID','TeamName':'LTeamName'}), on='LTeamID', how='left')

# Frame as higher seed vs lower seed
games_mania['HigherSeedWins'] = (games_mania['WSeed'] <= games_mania['LSeed']).astype(int)
games_mania['HigherTeamID'] = np.where(games_mania['WSeed'] <= games_mania['LSeed'], games_mania['WTeamID'],   games_mania['LTeamID'])
games_mania['LowerTeamID'] = np.where(games_mania['WSeed'] <= games_mania['LSeed'], games_mania['LTeamID'],   games_mania['WTeamID'])
games_mania['HigherSeedNum'] = games_mania[['WSeed','LSeed']].min(axis=1)
games_mania['LowerSeedNum'] = games_mania[['WSeed','LSeed']].max(axis=1)
games_mania['HigherTeamName'] = np.where(games_mania['WSeed'] <= games_mania['LSeed'], games_mania['WTeamName'], games_mania['LTeamName'])
games_mania['LowerTeamName'] = np.where(games_mania['WSeed'] <= games_mania['LSeed'], games_mania['LTeamName'], games_mania['WTeamName'])

print(games_mania[['Season','HigherTeamName','HigherSeedNum','LowerTeamName','LowerSeedNum','HigherSeedWins']].head(5))

   Season HigherTeamName  HigherSeedNum     LowerTeamName  LowerSeedNum  \
0    2002          Siena             16         Alcorn St            16   
1    2002        Alabama              2       FL Atlantic            15   
2    2002        Arizona              3  UC Santa Barbara            14   
3    2002           Duke              1          Winthrop            16   
4    2002        Indiana              5              Utah            12   

   HigherSeedWins  
0               1  
1               1  
2               1  
3               1  
4               1  



## 4. Merge Features

We merge the DEV efficiency stats onto each game for both teams, then add win percentage and POM rankings. Name matching is required because the DEV dataset uses ESPN team names while the Mania dataset uses its own system

In [9]:
# DEV feature columns
dev_feature_cols = [
    'Adjusted Tempo', 'Raw Tempo', 'Adjusted Offensive Efficiency',
    'Raw Offensive Efficiency', 'Adjusted Defensive Efficiency',
    'Raw Defensive Efficiency', 'eFGPct', 'TOPct', 'ORPct', 'FTRate',
    'OffFT', 'Off2PtFG', 'Off3PtFG', 'DefFT', 'Def2PtFG', 'Def3PtFG',
    'Tempo', 'AdjTempo', 'OE', 'AdjOE', 'DE', 'AdjDE', 'AdjEM',
    'FG2Pct', 'FG3Pct', 'FTPct', 'BlockPct', 'OppFG2Pct', 'OppFG3Pct',
    'OppFTPct', 'OppBlockPct', 'FG3Rate', 'OppFG3Rate', 'ARate',
    'OppARate', 'StlRate', 'OppStlRate', 'Net Rating',
]
dev_slim = dev[['Season', 'Mapped ESPN Team Name'] + dev_feature_cols].copy()
dev_slim = dev_slim.rename(columns={'Mapped ESPN Team Name': 'TeamName'})

# Team name matching Massey names to match DEV ESPN names
massey_to_dev = {
    'Connecticut': 'UConn', 'Utah St': 'Utah State', 'Wright St': 'Wright State',
    'N Dakota St': 'North Dakota State', 'Colorado St': 'Colorado State',
    'Florida St': 'Florida State', 'Kansas St': 'Kansas State',
    'Oklahoma St': 'Oklahoma State', 'Oregon St': 'Oregon State',
    'Washington St': 'Washington State', 'Boise St': 'Boise State',
    'San Diego St': 'San Diego State', 'Arizona St': 'Arizona State',
    'Wichita St': 'Wichita State', 'FL Atlantic': 'Florida Atlantic',
    'FGCU': 'Florida Gulf Coast', 'F Dickinson': 'Fairleigh Dickinson',
    'G Washington': 'George Washington', 'Loyola-Chicago': 'Loyola Chicago',
    'Loyola MD': 'Loyola Maryland', 'Mississippi': 'Ole Miss',
    'Mississippi St': 'Mississippi State', 'Monmouth NJ': 'Monmouth',
    'Morehead St': 'Morehead State', "Mt St Mary's": "Mount St. Mary's",
    'Murray St': 'Murray State', 'NC A&T': 'North Carolina A&T',
    'Norfolk St': 'Norfolk State', 'S Illinois': 'Southern Illinois',
    'SUNY Albany': 'UAlbany', 'St Bonaventure': 'St. Bonaventure',
    "St Joseph's PA": "Saint Joseph's", 'TAM C. Christi': 'Texas A&M-Corpus Christi',
    'TX Southern': 'Texas Southern', 'UT San Antonio': 'UTSA',
    'WKU': 'Western Kentucky', 'Abilene Chr': 'Abilene Christian',
    'Alcorn St': 'Alcorn State', 'American Univ': 'American University',
    'Appalachian St': 'App State', 'Ark Little Rock': 'Little Rock',
    'Ark Pine Bluff': 'Arkansas Pine Bluff', 'Boston Univ': 'Boston University',
    'C Michigan': 'Central Michigan', 'CS Bakersfield': 'Cal State Bakersfield',
    'CS Fullerton': 'Cal State Fullerton', 'CS Northridge': 'Cal State Northridge',
    'Central Conn': 'Central Connecticut', 'Cleveland St': 'Cleveland State',
    'Coastal Car': 'Coastal Carolina', 'Col Charleston': 'Charleston',
    'Coppin St': 'Coppin State', 'Delaware St': 'Delaware State',
    'Detroit': 'Detroit Mercy', 'E Kentucky': 'Eastern Kentucky',
    'E Washington': 'Eastern Washington', 'ETSU': 'East Tennessee State',
    'Fresno St': 'Fresno State', 'Gardner Webb': 'Gardner-Webb',
    'Georgia St': 'Georgia State', 'IL Chicago': 'UIC',
    'Indiana St': 'Indiana State', 'Jackson St': 'Jackson State',
    'Jacksonville St': 'Jacksonville State', 'Kent': 'Kent State',
    'Kennesaw': 'Kennesaw State', 'LIU Brooklyn': 'LIU',
    'Long Beach St': 'Long Beach State', 'MS Valley St': 'Mississippi Valley State',
    'MTSU': 'Middle Tennessee', 'McNeese St': 'McNeese',
    'Miami OH': 'Miami (OH)', 'Montana St': 'Montana State',
    'N Colorado': 'Northern Colorado', 'N Kentucky': 'Northern Kentucky',
    'NC Central': 'North Carolina Central', 'NE Omaha': 'Omaha',
    'New Mexico St': 'New Mexico State', 'Northwestern LA': 'Northwestern State',
    'Penn St': 'Penn State', 'Portland St': 'Portland State',
    'S Carolina St': 'South Carolina State', 'S Dakota St': 'South Dakota State',
    'SE Missouri St': 'Southeast Missouri State', 'SF Austin': 'Stephen F. Austin',
    'SIUE': 'SIU Edwardsville', 'Sam Houston St': 'Sam Houston',
    'Southern Univ': 'Southern', 'St Francis PA': 'St. Francis (PA)',
    "St Peter's": "Saint Peter's", 'Alabama St': 'Alabama State',
    'W Michigan': 'Western Michigan', 'WI Green Bay': 'Green Bay',
    'WI Milwaukee': 'Milwaukee', 'Weber St': 'Weber State',
    'Penn': 'Pennsylvania', 'Prairie View': 'Prairie View A&M',
    'Queens NC': 'Queens University', "St John's": "St. John's",
    'St Louis': 'Saint Louis', "St Mary's CA": "Saint Mary's",
    'Tennessee St': 'Tennessee State', 'Michigan St': 'Michigan State',
    'Ohio St': 'Ohio State', 'Iowa St': 'Iowa State',
    'Miami FL': 'Miami', 'NC State': 'NC State',
    'Morgan St': 'Morgan State', 'Winthrop': 'Winthrop',
    'James Madison': 'James Madison', 'Radford': 'Radford',
}

# Reverse map so DEV uses Massey names for merging
dev_to_massey = {v: k for k, v in massey_to_dev.items()}
dev_slim['TeamName_massey'] = dev_slim['TeamName'].replace(dev_to_massey)

In [10]:
# Merge DEV stats for higher and lower seed
games_merged = games_mania.merge(
    dev_slim.rename(columns={'TeamName_massey':'HigherTeamName'}).add_suffix('_H')
            .rename(columns={'Season_H':'Season','HigherTeamName_H':'HigherTeamName'}),
    on=['Season','HigherTeamName'], how='left'
)
games_merged = games_merged.merge(
    dev_slim.rename(columns={'TeamName_massey':'LowerTeamName'}).add_suffix('_L')
            .rename(columns={'Season_L':'Season','LowerTeamName_L':'LowerTeamName'}),
    on=['Season','LowerTeamName'], how='left'
)

#Merge win records and POM rankings
games_merged = games_merged.merge(
    record.rename(columns={'TeamID':'HigherTeamID','WinPct':'HigherWinPct'}),
    on=['Season','HigherTeamID'], how='left'
)
games_merged = games_merged.merge(
    record.rename(columns={'TeamID':'LowerTeamID','WinPct':'LowerWinPct'}),
    on=['Season','LowerTeamID'], how='left'
)
games_merged = games_merged.merge(
    pom_ranks.rename(columns={'TeamID':'HigherTeamID','POM_Rank':'HigherPOM'}),
    on=['Season','HigherTeamID'], how='left'
)
games_merged = games_merged.merge(
    pom_ranks.rename(columns={'TeamID':'LowerTeamID','POM_Rank':'LowerPOM'}),
    on=['Season','LowerTeamID'], how='left'
)

key_cols = ['AdjEM_H','AdjEM_L','HigherWinPct','LowerWinPct','HigherPOM','LowerPOM']


## 5. Feature Engineering

For each stat we compute higher seed minus lower seed. We also include individual seed numbers and win percentages as standalone features which adds meaningful signal for cross-seed matchups.

In [12]:
# Drop rows with missing DEV stats (7 in total)
games_final = games_merged.dropna(subset=['AdjEM_H','AdjEM_L']).copy()

# Build difference features 
diff_cols = []
for col in dev_feature_cols:
    h_col, l_col = f'{col}_H', f'{col}_L'
    if h_col in games_final.columns and l_col in games_final.columns:
        diff_name = f'{col}_diff'
        games_final[diff_name] = games_final[h_col] - games_final[l_col]
        diff_cols.append(diff_name)

games_final['POM_diff'] = games_final['HigherPOM'] - games_final['LowerPOM']

# Individual features which are more informative than differencess alone
individual_cols = ['HigherSeedNum','LowerSeedNum','HigherWinPct','LowerWinPct','HigherPOM','LowerPOM']
all_features = diff_cols + ['POM_diff'] + individual_cols

# Drop features with more than 10% nulls, fill remaining with median 
null_pct = games_final[all_features].isna().mean()
all_features = [f for f in all_features if null_pct[f] < 0.10]
for col in all_features:
    games_final[col] = games_final[col].fillna(games_final[col].median())



## 6. Model Validation, LOO-CV, and Hyperparameter Tuning

We tune hyperparameters using RandomizedSearchCV with the last 3 years as a validation set, then validate the tuned model across all years.

In [14]:
X = games_final[all_features]
y = games_final['HigherSeedWins']
years_cv = sorted(games_final['Season'].unique())

# Hyperparameter tuning
param_dist = {
    'n_estimators':[300, 500, 750, 1000],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}
val_mask = games_final['Season'].isin([2023, 2024, 2025])
split_index = np.where(val_mask, 0, -1)
ps = PredefinedSplit(split_index)

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=40, scoring='neg_brier_score',
    cv=ps, random_state=42, n_jobs=-1, verbose=1
)
search.fit(X, y)
best_params = search.best_params_
print(f"Best params: {best_params}")

Fitting 1 folds for each of 40 candidates, totalling 40 fits
Best params: {'n_estimators': 1000, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 30}


In [15]:
# LOO-CV with tuned parameters
tuned_brier, correct, total = [], 0, 0
for test_year in years_cv:
    train_mask = games_final['Season'] != test_year
    test_mask = games_final['Season'] == test_year
    rf_t = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    rf_t.fit(X[train_mask], y[train_mask])
    probs = rf_t.predict_proba(X[test_mask])[:,1]
    preds = (probs >= 0.5).astype(int)
    tuned_brier.append(brier_score_loss(y[test_mask], probs))
    correct += (preds == y[test_mask]).sum()
    total += test_mask.sum()
    print(f"{test_year}  Brier: {tuned_brier[-1]:.4f}  Games: {test_mask.sum()}")

print(f"  FINAL MODEL PERFORMANCE (Leave-One-Year-Out CV)")
print(f"  Brier Score: {np.mean(tuned_brier):.4f}")
print(f"  Accuracy:    {correct/total:.1%}")

2002  Brier: 0.1789  Games: 64
2003  Brier: 0.1424  Games: 63
2004  Brier: 0.1420  Games: 64
2005  Brier: 0.1299  Games: 64
2006  Brier: 0.1284  Games: 64
2007  Brier: 0.1158  Games: 64
2008  Brier: 0.0996  Games: 64
2009  Brier: 0.1251  Games: 64
2010  Brier: 0.1517  Games: 62
2011  Brier: 0.1927  Games: 66
2012  Brier: 0.1661  Games: 66
2013  Brier: 0.1439  Games: 66
2014  Brier: 0.1755  Games: 67
2015  Brier: 0.1367  Games: 67
2016  Brier: 0.1394  Games: 67
2017  Brier: 0.1261  Games: 68
2018  Brier: 0.1436  Games: 66
2019  Brier: 0.1221  Games: 67
2021  Brier: 0.1400  Games: 66
2022  Brier: 0.1592  Games: 67
2023  Brier: 0.1595  Games: 67
2024  Brier: 0.1418  Games: 67
2025  Brier: 0.1062  Games: 67
  FINAL MODEL PERFORMANCE (Leave-One-Year-Out CV)
  Brier Score: 0.1420
  Accuracy:    81.4%



## 7. Train Final Model


We train the tuned model on all 2002–2025 data to maximize signal for 2026 predictions.

In [17]:
# Train final model on all historical data
rf_final = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
rf_final.fit(X, y)

# Feature importance
importances = pd.Series(rf_final.feature_importances_, index=all_features).sort_values(ascending=False)
print(f"\nTop 15 stats:")
print(importances.head(15))

# Save model
with open('rf_march_madness_2026.pkl', 'wb') as f:
    pickle.dump({'model': rf_final, 'features': all_features, 'params': best_params}, f)


Top 15 stats:
AdjEM_diff         0.153613
Net Rating_diff    0.105655
POM_diff           0.071528
HigherSeedNum      0.031801
HigherPOM          0.026479
OppFG3Pct_diff     0.022446
AdjOE_diff         0.021345
LowerPOM           0.020531
HigherWinPct       0.020312
ARate_diff         0.020253
AdjDE_diff         0.020246
OppFG2Pct_diff     0.019466
FG2Pct_diff        0.019179
StlRate_diff       0.018065
BlockPct_diff      0.018033
dtype: float64



## 8. Generate 2026 Predictions

The format requires the probability that the higher seed wins for every possible matchup between all 68 teams so we predict all combinations.

In [19]:
# 2026 stats lookup
dev_2026 = dev[dev['Season'] == 2026].copy().set_index('Mapped ESPN Team Name')

sub_to_dev = {
    'Iowa St': 'Iowa State', 'Michigan St': 'Michigan State',
    'Ohio St': 'Ohio State', 'Connecticut': 'UConn',
    "St John's": "St. John's", "St Mary's CA": "Saint Mary's",
    'St Louis': 'Saint Louis', 'NC State': 'NC State',
    'N Dakota St': 'North Dakota State', 'Prairie View': 'Prairie View A&M',
    'Queens NC': 'Queens University', 'Tennessee St': 'Tennessee State',
    'Utah St': 'Utah State', 'Wright St': 'Wright State',
    'Kennesaw': 'Kennesaw State', 'McNeese St': 'McNeese',
    'Miami FL': 'Miami', 'Miami OH': 'Miami (OH)',
    'LIU Brooklyn': 'LIU', 'Cal Baptist': 'California Baptist',
    'Penn': 'Pennsylvania', 'N Iowa': 'Northern Iowa',
    'SMU': 'SMU', 'TEX/NCST': 'NC State',
}

seeds_2026 = pd.read_csv(os.path.join(mania_path, 'MNCAATourneySeeds.csv'))
seeds_2026 = seeds_2026[seeds_2026['Season'] == 2026].merge(mteams[['TeamID','TeamName']], on='TeamID')
massey_to_teamid = dict(zip(seeds_2026['TeamName'], seeds_2026['TeamID']))

def get_dev_stats(sub_name):
    dev_name = sub_to_dev.get(sub_name, sub_name)
    if dev_name in dev_2026.index: return dev_2026.loc[dev_name]
    if sub_name in dev_2026.index: return dev_2026.loc[sub_name]
    return dev_2026.mean(numeric_only=True)

def get_winpct(sub_name):
    team_id = massey_to_teamid.get(sub_name)
    if team_id is None:
        for k, v in massey_to_dev.items():
            if v == sub_to_dev.get(sub_name, sub_name):
                team_id = massey_to_teamid.get(k); break
    if team_id:
        rec = record[(record['Season'] == 2026) & (record['TeamID'] == team_id)]
        if len(rec) > 0: return rec.iloc[0]['WinPct']
    return 0.75

def get_pom(sub_name):
    team_id = massey_to_teamid.get(sub_name)
    if team_id is None:
        for k, v in massey_to_dev.items():
            if v == sub_to_dev.get(sub_name, sub_name):
                team_id = massey_to_teamid.get(k); break
    if team_id:
        pom = pom_ranks[(pom_ranks['Season'] == 2026) & (pom_ranks['TeamID'] == team_id)]
        if len(pom) > 0: return pom.iloc[0]['POM_Rank']
    return 100

def predict_matchup(higher_name, higher_seed_num, lower_name, lower_seed_num):
    stats_h, stats_l = get_dev_stats(higher_name), get_dev_stats(lower_name)
    row = {}
    for col in dev_feature_cols:
        diff_name = f'{col}_diff'
        if diff_name in all_features:
            row[diff_name] = stats_h.get(col, 0) - stats_l.get(col, 0)
    pom_h, pom_l = get_pom(higher_name), get_pom(lower_name)
    row['POM_diff'] = pom_h - pom_l
    row['HigherPOM'] = pom_h
    row['LowerPOM'] = pom_l
    row['HigherSeedNum'] = higher_seed_num
    row['LowerSeedNum'] = lower_seed_num
    row['HigherWinPct'] = get_winpct(higher_name)
    row['LowerWinPct'] = get_winpct(lower_name)
    return rf_final.predict_proba(pd.DataFrame([row])[all_features])[0][1]

#Generate all predictions
predictions = []
for _, row in df_sub.iterrows():
    prob = predict_matchup(row['HigherSeed'], row['HigherSeedNum'], row['LowerSeed'], row['LowerSeedNum'])
    predictions.append(round(prob, 4))

df_sub['Predictions'] = predictions
submission = df_sub[['HigherSeed','LowerSeed','Predictions']]
submission.to_csv('submission_2026.csv', index=False)

---
## 9. 2026 Bracket Simulation

We simulate each round using our model's probabilities, always advancing the team with probability > 50%.

In [21]:
bracket_2026 = {
    'East': [
        (1,'Duke'),(16,'Siena'),(8,'Ohio St'),(9,'TCU'),
        (5,"St John's"),(12,'N Iowa'),(4,'Kansas'),(13,'Cal Baptist'),
        (6,'Louisville'),(11,'South Florida'),(3,'Michigan St'),(14,'N Dakota St'),
        (7,'UCLA'),(10,'UCF'),(2,'Connecticut'),(15,'Furman'),
    ],
    'South': [
        (1,'Florida'),(16,'Lehigh'),(8,'Clemson'),(9,'Iowa'),
        (5,'Vanderbilt'),(12,'McNeese St'),(4,'Nebraska'),(13,'Troy'),
        (6,'North Carolina'),(11,'VCU'),(3,'Illinois'),(14,'Penn'),
        (7,"St Mary's CA"),(10,'Texas A&M'),(2,'Houston'),(15,'Idaho'),
    ],
    'West': [
        (1,'Arizona'),(16,'LIU Brooklyn'),(8,'Villanova'),(9,'Utah St'),
        (5,'Wisconsin'),(12,'High Point'),(4,'Arkansas'),(13,'Hawaii'),
        (6,'BYU'),(11,'NC State'),(3,'Gonzaga'),(14,'Kennesaw'),
        (7,'Miami FL'),(10,'Missouri'),(2,'Purdue'),(15,'Queens NC'),
    ],
    'Midwest': [
        (1,'Michigan'),(16,'UMBC'),(8,'Georgia'),(9,'St Louis'),
        (5,'Texas Tech'),(12,'Akron'),(4,'Alabama'),(13,'Hofstra'),
        (6,'Tennessee'),(11,'SMU'),(3,'Virginia'),(14,'Wright St'),
        (7,'Kentucky'),(10,'Santa Clara'),(2,'Iowa St'),(15,'Tennessee St'),
    ],
}

round_names = {64:'Round of 64',32:'Round of 32',16:'Sweet 16',8:'Elite 8',4:'Final Four',2:'Championship'}
all_bracket_results = []

def predict_game(team_a, seed_a, team_b, seed_b):
    if seed_a <= seed_b:
        higher, h_seed, lower, l_seed = team_a, seed_a, team_b, seed_b
    else:
        higher, h_seed, lower, l_seed = team_b, seed_b, team_a, seed_a
    prob_higher = predict_matchup(higher, h_seed, lower, l_seed)
    prob_a  = prob_higher if seed_a <= seed_b else 1 - prob_higher
    winner  = team_a if prob_a >= 0.5 else team_b
    w_seed  = seed_a if winner == team_a else seed_b
    return prob_a, winner, w_seed

def simulate_region(region_name, teams):
    print(f"\n{'='*55}\n  {region_name.upper()}\n{'='*55}")
    current = list(teams)
    round_size = 64
    while len(current) > 1:
        next_round = []
        print(f"\n {round_names[round_size]} ")
        for i in range(0, len(current), 2):
            seed_a, team_a = current[i]
            seed_b, team_b = current[i+1]
            prob_a, winner, winner_seed = predict_game(team_a, seed_a, team_b, seed_b)
            prob_w = prob_a if winner == team_a else 1 - prob_a
            print(f"  ({seed_a}) {team_a:20s} vs ({seed_b}) {team_b:20s} → ({winner_seed}) {winner} [{prob_w:.1%}]")
            all_bracket_results.append({
                'region': region_name, 'round': round_names[round_size],
                'team_a': team_a, 'seed_a': seed_a, 'team_b': team_b, 'seed_b': seed_b,
                'prob_a': round(prob_a, 3), 'predicted_winner': winner, 'winner_seed': winner_seed
            })
            next_round.append((winner_seed, winner))
        current = next_round
        round_size //= 2
    return current[0]

final_four = []
for region, teams in bracket_2026.items():
    winner = simulate_region(region, teams)
    final_four.append((winner[0], winner[1], region))
    print(f"\n  >>> {region} Champion: ({winner[0]}) {winner[1]}")

print(f"\n{'='*55}\n  FINAL FOUR\n{'='*55}")
ff_games = [(final_four[0], final_four[1]), (final_four[2], final_four[3])]
championship = []
for (seed_a, team_a, reg_a), (seed_b, team_b, reg_b) in ff_games:
    prob_a, winner, winner_seed = predict_game(team_a, seed_a, team_b, seed_b)
    prob_w = prob_a if winner == team_a else 1 - prob_a
    print(f"\n  ({seed_a}) {team_a} [{reg_a}] vs ({seed_b}) {team_b} [{reg_b}]")
    print(f"  → ({winner_seed}) {winner} [{prob_w:.1%}]")
    all_bracket_results.append({
        'region': f'{reg_a}/{reg_b}', 'round': 'Final Four',
        'team_a': team_a, 'seed_a': seed_a, 'team_b': team_b, 'seed_b': seed_b,
        'prob_a': round(prob_a, 3), 'predicted_winner': winner, 'winner_seed': winner_seed
    })
    championship.append((winner_seed, winner))

print(f"\n{'='*55}\n  NATIONAL CHAMPIONSHIP\n{'='*55}")
(seed_a, team_a), (seed_b, team_b) = championship
prob_a, winner, winner_seed = predict_game(team_a, seed_a, team_b, seed_b)
prob_w = prob_a if winner == team_a else 1 - prob_a
print(f"\n  ({seed_a}) {team_a} vs ({seed_b}) {team_b}")
print(f"  → NATIONAL CHAMPION: ({winner_seed}) {winner} [{prob_w:.1%}]")
all_bracket_results.append({
    'region': 'National', 'round': 'Championship',
    'team_a': team_a, 'seed_a': seed_a, 'team_b': team_b, 'seed_b': seed_b,
    'prob_a': round(prob_a, 3), 'predicted_winner': winner, 'winner_seed': winner_seed
})

pd.DataFrame(all_bracket_results).to_csv('2026_bracket_predictions.csv', index=False)
print(f"\nSaved 2026_bracket_predictions.csv ({len(all_bracket_results)} games)")


  EAST

  --- Round of 64 ---
  (1) Duke                 vs (16) Siena                → (1) Duke [98.7%]
  (8) Ohio St              vs (9) TCU                  → (8) Ohio St [51.6%]
  (5) St John's            vs (12) N Iowa               → (5) St John's [88.9%]
  (4) Kansas               vs (13) Cal Baptist          → (4) Kansas [90.3%]
  (6) Louisville           vs (11) South Florida        → (6) Louisville [86.7%]
  (3) Michigan St          vs (14) N Dakota St          → (3) Michigan St [95.1%]
  (7) UCLA                 vs (10) UCF                  → (7) UCLA [75.2%]
  (2) Connecticut          vs (15) Furman               → (2) Connecticut [98.6%]

  --- Round of 32 ---
  (1) Duke                 vs (8) Ohio St              → (1) Duke [97.2%]
  (5) St John's            vs (4) Kansas               → (5) St John's [52.3%]
  (6) Louisville           vs (3) Michigan St          → (3) Michigan St [64.0%]
  (7) UCLA                 vs (2) Connecticut          → (2) Connecticut [68.6%]

 